Install Packages

In [2]:
#!/usr/bin/env python3
import os
import time
from multiprocessing import cpu_count
from typing import Union, NamedTuple

import torch
import torch.backends.cudnn
import numpy as np
from torch import nn, optim
from torch.nn import functional as F
import torchvision.datasets
from torch.optim.optimizer import Optimizer
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms

In [3]:
from dataset import MIT

In [4]:
torch.backends.cudnn.benchmark = True

In [5]:
train_dataset_path = './train_data.pth.tar'
val_dataset_path = './val_data.pth.tar'
test_dataset_path = './test_data.pth.tar'

In [6]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

In [ ]:
class ImageShape(NamedTuple):
    height: int
    width: int
    channels: int

In [8]:
class MrCNN(nn.Module):
    def __init__(self, height: int, width: int, channels: int, class_count: int):
        super().__init__()
        self.input_shape = ImageShape(height=height, width=width, channels=channels)
        self.class_count = class_count

        self.conv1 = nn.Conv2d(
            in_channels=self.input_shape.channels,
            out_channels=96,
            kernel_size=(7, 7),
        )
        self.initialise_layer(self.conv1)
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2), stride=(1, 1))
        
        self.conv2 = nn.Conv2d(
            in_channels=96,  # Change to 32
            out_channels=160,
            kernel_size=(3, 3),
        )
        self.initialise_layer(self.conv2)
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2), stride=(1, 1))

        self.conv3 = nn.Conv2d(
            in_channels=160,  # Change to 64
            out_channels=288,
            kernel_size=(3, 3)
        )
        self.initialise_layer(self.conv3)
        self.pool3 = nn.MaxPool2d(kernel_size=(2, 2), stride=(1, 1))

        self.pool3 = torch.flatten(self.pool3, start_dim = 1)

        self.fc1 = nn.Linear(2592, 512)
        self.initialise_layer(self.fc1)
        
        self.fc2 = nn.Linear(512, 1)
        self.initialise_layer(self.fc2)
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(images))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))  
        x = self.pool2(x)
        x = F.relu(self.conv3(x))  
        x = self.pool3(x)
        x = torch.flatten(x,start_dim=1)
        x = F.relu(self.fc1(x))
        
        x = self.fc2(x)
        return x

    @staticmethod
    def initialise_layer(layer):
        if hasattr(layer, "bias"):
            nn.init.zeros_(layer.bias)
        if hasattr(layer, "weight"):
            nn.init.kaiming_normal_(layer.weight)

In [9]:
train_dataset = MIT(dataset_path=train_dataset_path).dataset
test_dataset = MIT(dataset_path=test_dataset_path).dataset
val_dataset = MIT(dataset_path=val_dataset_path).dataset

In [10]:
train_loader = torch.utils.data.DataLoader(
        train_dataset,
        shuffle=True,
        batch_size=10,
        pin_memory=True,
        num_workers=1,
    )
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    shuffle=False,
    batch_size=10,
    num_workers=1,
    pin_memory=True,
    )

In [38]:
def compute_accuracy(labels, preds):
    correct = (labels == preds).sum().item()
    total = labels.size(0)
    return correct / total

In [40]:
test_loader.dataset[0].keys()

dict_keys(['X', 'X_400', 'X_250', 'X_150', 'y', 'spatial_coords', 'file'])

In [47]:
test_loader.dataset[0]["X"].shape

torch.Size([3, 683, 1024])

In [48]:
test_loader.dataset[0]["X_400"].shape

torch.Size([3, 400, 400])

In [49]:
class Trainer:
    def __init__(
        self,
        model: nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        criterion: nn.Module,
        optimizer: Optimizer,
        summary_writer: SummaryWriter,
        device: torch.device,
    ):
        self.model = model.to(device)
        self.device = device
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.summary_writer = summary_writer
        self.step = 0

    def train(
        self,
        epochs: int,
        val_frequency: int,
        print_frequency: int = 20,
        log_frequency: int = 5,
        start_epoch: int = 0
    ):
        self.model.train()
        for epoch in range(start_epoch, epochs):
            self.model.train()
            data_load_start_time = time.time()
            for index in range(0, batch_size = 10):
                batch = self.train_loader.dataset[index]["X"]
                label = self.train_loader.dataset[index]["y"]
                batch = batch.to(self.device)
                label = label.to(self.device)
                data_load_end_time = time.time()


                ## TASK 1: Compute the forward pass of the model, print the output shape
                ##         and quit the program
                #output =
                output = self.model.forward(batch)
                #print(output.shape)
                #import sys; sys.exit(1)
                ## TASK 7: Rename `output` to `logits`, remove the output shape printing
                ##         and get rid of the `import sys; sys.exit(1)`
                logits = self.model.forward(batch)
                ## TASK 9: Compute the loss using self.criterion and
                ##         store it in a variable called `loss`
                loss = self.criterion(logits, label)
                #print(f"Epoch [{epoch + 1}/{epochs}], Batch Loss: {loss.item()}")
                ## TASK 10: Compute the backward pass
                loss.backward()
                ## TASK 12: Step the optimizer and then zero out the gradient buffers.
                self.optimizer.step()
                self.optimizer.zero_grad()
                
                with torch.no_grad():
                    preds = logits.argmax(-1)
                    accuracy = compute_accuracy(label, preds)

                data_load_time = data_load_end_time - data_load_start_time
                step_time = time.time() - data_load_end_time
                if ((self.step + 1) % log_frequency) == 0:
                    self.log_metrics(epoch, accuracy, loss, data_load_time, step_time)
                if ((self.step + 1) % print_frequency) == 0:
                    self.print_metrics(epoch, accuracy, loss, data_load_time, step_time)

                self.step += 1
                data_load_start_time = time.time()

            self.summary_writer.add_scalar("epoch", epoch, self.step)
            if ((epoch + 1) % val_frequency) == 0:
                self.validate()
                # self.validate() will put the model in validation mode,
                # so we have to switch back to train mode afterwards
                self.model.train()

    def print_metrics(self, epoch, accuracy, loss, data_load_time, step_time):
        epoch_step = self.step % len(self.train_loader)
        print(
                f"epoch: [{epoch}], "
                f"step: [{epoch_step}/{len(self.train_loader)}], "
                f"batch loss: {loss:.5f}, "
                f"batch accuracy: {accuracy * 100:2.2f}, "
                f"data load time: "
                f"{data_load_time:.5f}, "
                f"step time: {step_time:.5f}"
        )

    def log_metrics(self, epoch, accuracy, loss, data_load_time, step_time):
        self.summary_writer.add_scalar("epoch", epoch, self.step)
        self.summary_writer.add_scalars(
                "accuracy",
                {"train": accuracy},
                self.step
        )
        self.summary_writer.add_scalars(
                "loss",
                {"train": float(loss.item())},
                self.step
        )
        self.summary_writer.add_scalar(
                "time/data", data_load_time, self.step
        )
        self.summary_writer.add_scalar(
                "time/data", step_time, self.step
        )

    def validate(self):
        results = {"preds": [], "labels": []}
        total_loss = 0
        self.model.eval()

        # No need to track gradients for validation, we're not optimizing.
        with torch.no_grad():
            for index in range(0, batch_size = 10):
                batch = self.val_loader.dataset[index]["X_400"]
                label = self.val_loader.dataset[index]["y"]
                batch = batch.to(self.device)
                label = label.to(self.device)
                logits = self.model(batch)
                loss = self.criterion(logits, label)
                total_loss += loss.item()
                preds = logits.argmax(dim=-1).cpu().numpy()
                results["preds"].extend(list(preds))
                results["labels"].extend(list(label.cpu().numpy()))

        accuracy = compute_accuracy(
            np.array(results["labels"]), np.array(results["preds"])
        )
        average_loss = total_loss / len(self.val_loader)

        self.summary_writer.add_scalars(
                "accuracy",
                {"test": accuracy},
                self.step
        )
        self.summary_writer.add_scalars(
                "loss",
                {"test": average_loss},
                self.step
        )
        print(f"validation loss: {average_loss:.5f}, accuracy: {accuracy * 100:2.2f}")

In [50]:
import argparse
from pathlib import Path
parser = argparse.ArgumentParser(
    description="Train a MRCNN on MIT dataset",
    formatter_class=argparse.ArgumentDefaultsHelpFormatter,
)
default_dataset_dir = Path.home() / ".cache" / "torch" / "datasets"
parser.add_argument("--dataset-root", default=default_dataset_dir)
parser.add_argument("--log-dir", default=Path("logs"), type=Path)

_StoreAction(option_strings=['--log-dir'], dest='log_dir', nargs=None, const=None, default=PosixPath('logs'), type=<class 'pathlib.Path'>, choices=None, required=False, help=None, metavar=None)

In [ ]:
def get_summary_writer_log_dir(args: argparse.Namespace) -> str:
    """Get a unique directory that hasn't been logged to before for use with a TB
    SummaryWriter.

    Args:
        args: CLI Arguments

    Returns:
        Subdirectory of log_dir with unique subdirectory name to prevent multiple runs
        from getting logged to the same TB log directory (which you can't easily
        untangle in TB).
    """
    tb_log_dir_prefix = f'CNN_bs={args.batch_size}_lr={args.learning_rate}_run_'
    i = 0
    while i < 1000:
        tb_log_dir = args.log_dir / (tb_log_dir_prefix + str(i))
        if not tb_log_dir.exists():
            return str(tb_log_dir)
        i += 1
    return str(tb_log_dir)

In [52]:
def main(args):
    
    train_dataset = MIT(dataset_path=train_dataset_path).dataset
    test_dataset = MIT(dataset_path=test_dataset_path).dataset
    val_dataset = MIT(dataset_path=val_dataset_path).dataset
    
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        shuffle=True,
        batch_size=args.batch_size,
        pin_memory=True,
        num_workers=args.worker_count,
    )
    test_loader = torch.utils.data.DataLoader(
        test_dataset,
        shuffle=False,
        batch_size=args.batch_size,
        num_workers=args.worker_count,
        pin_memory=True,
    )
    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        shuffle=False,
        batch_size=args.batch_size,
        num_workers=args.worker_count,
        pin_memory=True,
    )

    model = MrCNN(input_channels=3, output_classes=1)

    ## TASK 8: Redefine the criterion to be softmax cross entropy
    criterion = nn.BCEWithLogitsLoss()

    ## TASK 11: Define the optimizer
    optimizer = torch.optim.SGD(model.parameters(), lr=args.learning_rate)

    log_dir = get_summary_writer_log_dir(args)
    print(f"Writing logs to {log_dir}")
    summary_writer = SummaryWriter(
            str(log_dir),
            flush_secs=5
    )
    trainer = Trainer(
        model, train_loader, test_loader, criterion, optimizer, summary_writer, DEVICE
    )

    trainer.train(
        args.epochs,
        args.val_frequency,
        print_frequency=args.print_frequency,
        log_frequency=args.log_frequency,
    )

    summary_writer.close()

In [ ]:
if __name__ == "__main__":
    main(parser.parse_args())   

usage: ipykernel_launcher.py [-h] [--dataset-root DATASET_ROOT]
                             [--log-dir LOG_DIR]
ipykernel_launcher.py: error: unrecognized arguments: --f=/Users/smaranda/Library/Jupyter/runtime/kernel-v36d20bb9a6f4fca704a51085939a926a348e8ebff.json


SystemExit: 2

/Users/smaranda/anaconda3/envs/COMS30035_labs/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
